# 🧬 Bioinformatik-App

**So geht's:** Bei jeder grauen Box links auf ▶ klicken (oder hineinklicken und **Shift + Enter**). Die Felder rechts in der Box kannst du vorher ändern: Auswahllisten, Textfelder, Häkchen. Den Code dahinter musst du nicht anfassen (Doppelklick auf eine Box zeigt ihn, falls du neugierig bist).

**Fang mit Box 0 an.** Sie lädt die neueste Version der Werkzeuge. Danach kannst du jede Box einzeln benutzen.

| Box | Was sie macht |
|---|---|
| 0 ⚙️ | Werkzeuge laden bzw. aktualisieren |
| 1 📊 | **Hauptbox:** Projekte auswerten. Du wählst die Analyse (alles, Protein-Vergleich, 3D-Struktur, Stammbaum, Mutationen …) → Bericht mit Tabellen und Diagrammen |
| 2 📤 | Eigene FASTA-Dateien hochladen |
| 3 🧬 | Eine Sequenz umwandeln: DNA, mRNA, Gegenstrang, Protein |
| 4 🔍 | ORFs (mögliche Gene) suchen, mit Karte |
| 5 ⚖️ | Zwei DNA-Sequenzen vergleichen: Alignment, Mutationen, Wirkung aufs Protein |
| 6 🧪 | Proteine vergleichen (BLOSUM62 wie BLASTP) und Eigenschaften berechnen: Gewicht, pI, Ladung, Hydropathie, Instabilitätsindex |
| 7 🧊 | 3D-Struktur: Bindestelle (z. B. Häm), Lage einer Mutation (innen oder außen), 3D-Ansicht |
| 8 🩸 | Mutations-Labor: selbst eine Mutation ins Hämoglobin-Gen einbauen |
| 9 🌐 | Sequenzen von NCBI herunterladen |

In [ ]:
#@title 0 ⚙️ Werkzeuge laden (zuerst ausführen)
#@markdown Holt die neueste Version aus GitHub. Wieder ausführen, wenn es ein Update gab.
GOOGLE_DRIVE_VERBINDEN = False #@param {type:"boolean"}

import os, sys, subprocess
if not os.path.exists("Bioinfo/.git"):
    subprocess.run(["git", "clone", "-q", "https://github.com/galaxytosst/Bioinfo"], check=True)
else:
    subprocess.run(["git", "-C", "Bioinfo", "fetch", "-q", "origin", "main"], check=True)
    subprocess.run(["git", "-C", "Bioinfo", "reset", "-q", "--hard", "origin/main"], check=True)
if os.path.abspath("Bioinfo") not in sys.path:
    sys.path.insert(0, os.path.abspath("Bioinfo"))
for modul in ("werkzeuge", "analyse", "bericht", "oberflaeche", "struktur"):
    sys.modules.pop(modul, None)
import werkzeuge, analyse, bericht, oberflaeche as O
stand = subprocess.run(["git", "-C", "Bioinfo", "log", "-1", "--format=%h vom %cd", "--date=format:%d.%m.%Y %H:%M"],
                       capture_output=True, text=True).stdout.strip()
if GOOGLE_DRIVE_VERBINDEN:
    from google.colab import drive
    drive.mount("/content/drive")
print(f"✅ Werkzeuge geladen (Stand {stand}). Jetzt kannst du die anderen Boxen benutzen.")

In [ ]:
#@title 1 📊 Analyse: Projekte auswerten → Bericht mit Tabellen und Diagrammen
#@markdown **Welche Analyse?** „Alles“ macht jeden Schritt. Aus der Liste klicken **oder** selbst kombinieren, z. B. `Mutationen, Protein-Vergleich`. Die Übersicht (Länge, GC-Gehalt) ist immer dabei.
#@markdown **3D-Struktur** (auch in „Alles“): Für jedes Projekt, in dem ein Protein erkannt wurde, sucht das Programm in der Proteindatenbank PDB eine passende Struktur (braucht Internet, einige Sekunden je Projekt). Im Bericht: 3D-Ansicht, Bindestellen und ob die Unterschiede innen, außen oder am Liganden liegen.
ANALYSE = "Alles" #@param ["Alles", "Übersicht + ORFs (mögliche Gene)", "DNA-Vergleich (Alignment, Identität)", "Mutationen (Varianten und ihre Wirkung)", "Protein-Vergleich", "3D-Struktur", "Stammbaum", "Protein-Vergleich, Stammbaum", "Mutationen, Protein-Vergleich", "Mutationen, 3D-Struktur"] {allow-input: true}
#@markdown **Woher kommen die Daten?**
QUELLE = "Beispiele" #@param ["Beispiele", "Eigene Daten (Colab-Ordner daten)", "Google Drive"]
DRIVE_ORDNER = "/content/drive/MyDrive/Bioinformatik/daten" #@param {type:"string"}
#@markdown **Welche Projekte?** Aus der Liste klicken **oder** selbst tippen: mehrere mit Komma (z. B. `01, 02`), eigene Projekte mit ihrem Ordnernamen
PROJEKTE = "alle" #@param ["alle", "01_sichelzellanaemie", "02_haemoglobin_arten", "03_coronaviren", "04_frauenschuh_orchideen", "05_pest_plasmid", "06_chloroplast_arabidopsis", "01, 02", "mein_projekt"] {allow-input: true}
#@markdown **Einstellungen** (leer bzw. „wie im Projekt“ = Standard)
ORF_MINDESTLAENGE = "wie im Projekt" #@param ["wie im Projekt", "30", "60", "100", "150", "300"]
STARTCODONS = "wie im Projekt" #@param ["wie im Projekt", "nur ATG", "ATG, GTG, TTG (Bakterien)"]
#@markdown Nur wenn genau ein Projekt gewählt ist: Referenz für die Variantenanalyse und Außengruppe (Namen mit Komma)
REFERENZ = "" #@param {type:"string"}
AUSSENGRUPPE = "" #@param {type:"string"}
#@markdown 3D-Struktur: „wie im Projekt“ = Eintrag aus einstellungen.json, sonst automatische Suche. Eine PDB-ID (z. B. `4HHB`) oder UniProt-Nr. (z. B. `P68871`, dann AlphaFold-Modell) gilt nur, wenn genau ein Projekt gewählt ist.
STRUKTUR = "wie im Projekt" #@param ["wie im Projekt", "automatisch suchen", "4HHB", "P68871"] {allow-input: true}
BERICHT_HIER_ANZEIGEN = True #@param {type:"boolean"}

# --- Werkzeuge bereitstellen (falls Zelle 0 noch nicht lief) ---
import os, sys, subprocess
if not os.path.exists("Bioinfo/.git"):
    subprocess.run(["git", "clone", "-q", "https://github.com/galaxytosst/Bioinfo"], check=True)
if os.path.abspath("Bioinfo") not in sys.path:
    sys.path.insert(0, os.path.abspath("Bioinfo"))
import oberflaeche as O, analyse
from IPython.display import HTML, display

def _ziel_basis(ziel, drive_ordner):
    if ziel == "Google Drive":
        if not os.path.isdir("/content/drive/MyDrive"):
            from google.colab import drive
            drive.mount("/content/drive")
        return drive_ordner.rstrip("/")
    return os.path.abspath("daten")

if QUELLE == "Beispiele":
    basis, ausgabe = O.BEISPIELE, "ergebnisse_beispiele"
elif QUELLE == "Google Drive":
    basis = _ziel_basis("Google Drive", DRIVE_ORDNER)
    ausgabe = basis + "_ergebnisse"
else:
    basis, ausgabe = os.path.abspath("daten"), "ergebnisse"
os.makedirs(basis, exist_ok=True)

alle_projekte = analyse.projekte_im_ordner(basis, ausgabe)
alle_projekte = [p for p in alle_projekte if os.listdir(p)]
namen = {os.path.basename(p): p for p in alle_projekte}
print("Vorhandene Projekte:", ", ".join(namen) if namen else "keine")
if not namen:
    print(f"Der Ordner {basis} ist leer. Lade erst Dateien hoch (Box 2) oder hol sie von NCBI (Box 9).")
else:
    if PROJEKTE.strip().lower() in ("", "alle"):
        gewaehlt = list(namen)
    else:
        gewaehlt = []
        for wunsch in [w.strip() for w in PROJEKTE.split(",") if w.strip()]:
            treffer = [n for n in namen if n == wunsch or n.startswith(wunsch)]
            if not treffer:
                raise ValueError(f"Projekt '{wunsch}' gibt es nicht. Vorhanden: {', '.join(namen)}")
            gewaehlt += [t for t in treffer if t not in gewaehlt]
    einstellungen = {"SCHRITTE": ANALYSE}
    schritte = analyse.schritte_aus_text(ANALYSE)
    print("Analyse:", ", ".join(analyse.SCHRITT_NAMEN[s] for s in analyse.ALLE_SCHRITTE if s in schritte))
    if ORF_MINDESTLAENGE != "wie im Projekt":
        einstellungen["ORF_MIN_AMINOSAEUREN"] = int(ORF_MINDESTLAENGE)
    if STARTCODONS == "nur ATG":
        einstellungen["STARTCODONS"] = ("ATG",)
    elif STARTCODONS.startswith("ATG, GTG"):
        einstellungen["STARTCODONS"] = ("ATG", "GTG", "TTG")
    if REFERENZ.strip() or AUSSENGRUPPE.strip():
        if len(gewaehlt) != 1:
            print("⚠️ REFERENZ und AUSSENGRUPPE gelten nur, wenn genau ein Projekt gewählt ist. Sie werden ignoriert.")
        else:
            if REFERENZ.strip():
                einstellungen["REFERENZ"] = REFERENZ.strip()
            if AUSSENGRUPPE.strip():
                einstellungen["AUSSENGRUPPE"] = [a.strip() for a in AUSSENGRUPPE.split(",") if a.strip()]
    if STRUKTUR.strip() == "automatisch suchen":
        einstellungen["STRUKTUR"] = "automatisch"
    elif STRUKTUR.strip() not in ("", "wie im Projekt"):
        if len(gewaehlt) != 1:
            print("⚠️ Eine feste STRUKTUR gilt nur, wenn genau ein Projekt gewählt ist. Sie wird ignoriert.")
        else:
            einstellungen["STRUKTUR"] = STRUKTUR.strip()
    if "struktur" in schritte:
        print("3D-Struktur: braucht Internet (rcsb.org, UniProt). Das kann je Projekt einige Sekunden dauern.")
    import shutil
    if os.path.isdir(ausgabe):
        shutil.rmtree(ausgabe)
    ergebnisse, bericht_pfad = analyse.analysiere_projekte([namen[g] for g in gewaehlt], ausgabe, **einstellungen)
    display(HTML(O.bericht_html(bericht_pfad, BERICHT_HIER_ANZEIGEN)))

In [ ]:
#@title 2 📤 Eigene Dateien hochladen (werden ein neues Projekt)
#@markdown Nach dem Start erscheint ein Knopf **„Dateien auswählen“**. FASTA-Dateien wählen (auch mehrere). Danach in Box 1 bei QUELLE die gleiche Stelle auswählen.
PROJEKTNAME = "mein_projekt" #@param {type:"string"}
ZIEL = "Eigene Daten (Colab-Ordner daten)" #@param ["Eigene Daten (Colab-Ordner daten)", "Google Drive"]
DRIVE_ORDNER = "/content/drive/MyDrive/Bioinformatik/daten" #@param {type:"string"}

# --- Werkzeuge bereitstellen (falls Zelle 0 noch nicht lief) ---
import os, sys, subprocess
if not os.path.exists("Bioinfo/.git"):
    subprocess.run(["git", "clone", "-q", "https://github.com/galaxytosst/Bioinfo"], check=True)
if os.path.abspath("Bioinfo") not in sys.path:
    sys.path.insert(0, os.path.abspath("Bioinfo"))
import oberflaeche as O, analyse
from IPython.display import HTML, display

def _ziel_basis(ziel, drive_ordner):
    if ziel == "Google Drive":
        if not os.path.isdir("/content/drive/MyDrive"):
            from google.colab import drive
            drive.mount("/content/drive")
        return drive_ordner.rstrip("/")
    return os.path.abspath("daten")

from google.colab import files
ziel = os.path.join(_ziel_basis(ZIEL, DRIVE_ORDNER), O._projektname(PROJEKTNAME))
os.makedirs(ziel, exist_ok=True)
hochgeladen = files.upload()
for dateiname, inhalt in hochgeladen.items():
    with open(os.path.join(ziel, os.path.basename(dateiname)), "wb") as f:
        f.write(inhalt)
    print(f"✅ gespeichert: {ziel}/{os.path.basename(dateiname)} ({len(inhalt):,} Bytes)")
print(f"Jetzt Box 1 ausführen mit PROJEKTE = {O._projektname(PROJEKTNAME)}")

In [ ]:
#@title 3 🧬 Eine Sequenz umwandeln: DNA → mRNA → Protein
#@markdown Entweder ein Beispiel wählen **oder** unten eine eigene Sequenz einfügen (DNA oder RNA, auch FASTA). Ist SEQUENZ ausgefüllt, wird sie benutzt.
BEISPIEL = "HBB-mRNA des Menschen" #@param ["kurzes Testgen", "HBB-mRNA des Menschen", "HbS (Sichelzelle)", "HBB Maus", "Pest-Plasmid pPCP1"]
SEQUENZ = "" #@param {type:"string"}

# --- Werkzeuge bereitstellen (falls Zelle 0 noch nicht lief) ---
import os, sys, subprocess
if not os.path.exists("Bioinfo/.git"):
    subprocess.run(["git", "clone", "-q", "https://github.com/galaxytosst/Bioinfo"], check=True)
if os.path.abspath("Bioinfo") not in sys.path:
    sys.path.insert(0, os.path.abspath("Bioinfo"))
import oberflaeche as O, analyse
from IPython.display import HTML, display

eingabe = SEQUENZ if SEQUENZ.strip() else O.beispiel_fasta(BEISPIEL)
display(HTML(O.sequenz_html(eingabe)))

In [ ]:
#@title 4 🔍 ORFs suchen (mögliche Gene in allen 6 Leserastern)
BEISPIEL = "Pest-Plasmid pPCP1" #@param ["kurzes Testgen", "HBB-mRNA des Menschen", "Pest-Plasmid pPCP1"]
SEQUENZ = "" #@param {type:"string"}
MINDESTLAENGE_AMINOSAEUREN = 60 #@param {type:"slider", min:10, max:400, step:10}
STARTCODONS = "ATG, GTG, TTG (Bakterien)" #@param ["nur ATG", "ATG, GTG, TTG (Bakterien)"]

# --- Werkzeuge bereitstellen (falls Zelle 0 noch nicht lief) ---
import os, sys, subprocess
if not os.path.exists("Bioinfo/.git"):
    subprocess.run(["git", "clone", "-q", "https://github.com/galaxytosst/Bioinfo"], check=True)
if os.path.abspath("Bioinfo") not in sys.path:
    sys.path.insert(0, os.path.abspath("Bioinfo"))
import oberflaeche as O, analyse
from IPython.display import HTML, display

eingabe = SEQUENZ if SEQUENZ.strip() else O.beispiel_fasta(BEISPIEL)
codons = ("ATG",) if STARTCODONS == "nur ATG" else ("ATG", "GTG", "TTG")
display(HTML(O.orfs_html(eingabe, MINDESTLAENGE_AMINOSAEUREN, codons)))

In [ ]:
#@title 5 ⚖️ Zwei Sequenzen vergleichen (Alignment, Mutationen, Wirkung)
#@markdown Beispiel wählen **oder** zwei eigene Sequenzen einfügen (Sequenz 1 = Referenz).
BEISPIEL = "Sichelzelle: HbA gegen HbS" #@param ["Sichelzelle: HbA gegen HbS", "β-Thalassämie: HbA gegen Codon 41/42", "Mensch gegen Schimpanse (HBB)", "Mensch gegen Maus (HBB)"]
SEQUENZ_1 = "" #@param {type:"string"}
SEQUENZ_2 = "" #@param {type:"string"}
MODUS = "global (ganze Sequenzen)" #@param ["global (ganze Sequenzen)", "lokal (bester gemeinsamer Abschnitt)"]
GEGENSTRANG_PRUEFEN = True #@param {type:"boolean"}

# --- Werkzeuge bereitstellen (falls Zelle 0 noch nicht lief) ---
import os, sys, subprocess
if not os.path.exists("Bioinfo/.git"):
    subprocess.run(["git", "clone", "-q", "https://github.com/galaxytosst/Bioinfo"], check=True)
if os.path.abspath("Bioinfo") not in sys.path:
    sys.path.insert(0, os.path.abspath("Bioinfo"))
import oberflaeche as O, analyse
from IPython.display import HTML, display

paare = {"Sichelzelle: HbA gegen HbS": "HbS (Sichelzelle)",
         "β-Thalassämie: HbA gegen Codon 41/42": "β-Thalassämie Codon 41/42",
         "Mensch gegen Schimpanse (HBB)": "HBB Schimpanse", "Mensch gegen Maus (HBB)": "HBB Maus"}
if SEQUENZ_1.strip() and SEQUENZ_2.strip():
    s1, s2 = SEQUENZ_1, SEQUENZ_2
else:
    s1, s2 = O.beispiel_fasta("HBB-mRNA des Menschen"), O.beispiel_fasta(paare[BEISPIEL])
display(HTML(O.vergleich_html(s1, s2, MODUS.split()[0], GEGENSTRANG_PRUEFEN)))

In [ ]:
#@title 6 🧪 Proteine: vergleichen und Eigenschaften berechnen (Gewicht, pI, Ladung, Stabilität)
#@markdown Beispiel wählen **oder** eigene Sequenzen einfügen. **Beide Felder** ausgefüllt = Vergleich (Protein 1 = Referenz) plus Eigenschaften beider Proteine. **Nur PROTEIN_1** ausgefüllt = nur die Eigenschaften dieses Proteins. Erlaubt sind Proteinsequenzen (z. B. aus UniProt) **und** DNA/mRNA: dann wird das Gen (längster ORF ab ATG) automatisch übersetzt.
BEISPIEL = "Sichelzelle: HbA gegen HbS" #@param ["Sichelzelle: HbA gegen HbS", "Mensch gegen Schimpanse (HBB)", "Mensch gegen Maus (HBB)", "Mensch gegen Rind (HBB, mit Deletion)", "Nur Eigenschaften: Hämoglobin β (Mensch)", "Nur Eigenschaften: Protein aus dem Pest-Plasmid"]
PROTEIN_1 = "" #@param {type:"string"}
PROTEIN_2 = "" #@param {type:"string"}
MODUS = "global (ganze Proteine)" #@param ["global (ganze Proteine)", "lokal (bester gemeinsamer Abschnitt)"]

# --- Werkzeuge bereitstellen (falls Zelle 0 noch nicht lief) ---
import os, sys, subprocess
if not os.path.exists("Bioinfo/.git"):
    subprocess.run(["git", "clone", "-q", "https://github.com/galaxytosst/Bioinfo"], check=True)
if os.path.abspath("Bioinfo") not in sys.path:
    sys.path.insert(0, os.path.abspath("Bioinfo"))
import oberflaeche as O, analyse
from IPython.display import HTML, display

paare = {"Sichelzelle: HbA gegen HbS": "HbS (Sichelzelle)", "Mensch gegen Schimpanse (HBB)": "HBB Schimpanse",
         "Mensch gegen Maus (HBB)": "HBB Maus", "Mensch gegen Rind (HBB, mit Deletion)": "HBB Rind"}
einzeln = {"Nur Eigenschaften: Hämoglobin β (Mensch)": "HBB-mRNA des Menschen",
           "Nur Eigenschaften: Protein aus dem Pest-Plasmid": "Pest-Plasmid pPCP1"}
if PROTEIN_1.strip() and PROTEIN_2.strip():
    display(HTML(O.protein_vergleich_html(PROTEIN_1, PROTEIN_2, MODUS.split()[0])))
elif PROTEIN_1.strip() or PROTEIN_2.strip():
    display(HTML(O.protein_eigenschaften_html(PROTEIN_1 if PROTEIN_1.strip() else PROTEIN_2)))
elif BEISPIEL in einzeln:
    display(HTML(O.protein_eigenschaften_html(O.beispiel_fasta(einzeln[BEISPIEL]))))
else:
    s1, s2 = O.beispiel_fasta("HBB-mRNA des Menschen"), O.beispiel_fasta(paare[BEISPIEL])
    display(HTML(O.protein_vergleich_html(s1, s2, MODUS.split()[0])))

In [ ]:
#@title 7 🧊 3D-Struktur: Bindestelle, Lage einer Mutation, 3D-Ansicht
#@markdown **Beispiel** wählen oder „eigene Eingabe“ und die Felder darunter ausfüllen. Braucht Internet (lädt die Struktur von rcsb.org bzw. alphafold.ebi.ac.uk).
BEISPIEL = "Sichelzelle: Glu7Val im Hämoglobin (4HHB)" #@param ["Sichelzelle: Glu7Val im Hämoglobin (4HHB)", "Häm-Tasche: die beiden Histidine (4HHB)", "AlphaFold-Modell von Hämoglobin β (P68871)", "HIV-Protease: aktives Zentrum mit Hemmstoff (1HVR)", "eigene Eingabe (Felder unten)"]
#@markdown **STRUKTUR**: PDB-ID (z. B. `4HHB`, zu finden auf rcsb.org), UniProt-Nummer für ein AlphaFold-Modell (z. B. `P68871`) oder Pfad zu einer hochgeladenen .pdb-Datei
STRUKTUR = "4HHB" #@param {type:"string"}
#@markdown **PROTEIN**: auf welches Protein sich die Positionen beziehen. „keins“ = Nummern der Struktur. Du kannst auch eine Protein- oder DNA-Sequenz einfügen.
PROTEIN = "HBB-mRNA des Menschen" #@param ["HBB-mRNA des Menschen", "keins (Nummern der Struktur)"] {allow-input: true}
#@markdown **POSITIONEN**: z. B. `7`, `E7V` oder `Glu7Val, His64` (mehrere mit Komma)
POSITIONEN = "E7V" #@param {type:"string"}
#@markdown **UNIPROT** (optional): bekannte aktive Stellen, Bindestellen und Varianten, z. B. `P68871` für Hämoglobin β
UNIPROT = "P68871" #@param {type:"string"}
KONTAKT_ABSTAND = 4 #@param {type:"slider", min:3, max:6, step:0.5}

# --- Werkzeuge bereitstellen (falls Zelle 0 noch nicht lief) ---
import os, sys, subprocess
if not os.path.exists("Bioinfo/.git"):
    subprocess.run(["git", "clone", "-q", "https://github.com/galaxytosst/Bioinfo"], check=True)
if os.path.abspath("Bioinfo") not in sys.path:
    sys.path.insert(0, os.path.abspath("Bioinfo"))
import oberflaeche as O, analyse
from IPython.display import HTML, display

beispiele = {
    "Sichelzelle: Glu7Val im Hämoglobin (4HHB)": ("4HHB", "HBB-mRNA des Menschen", "E7V", "P68871"),
    "Häm-Tasche: die beiden Histidine (4HHB)": ("4HHB", "HBB-mRNA des Menschen", "His64, His93", "P68871"),
    "AlphaFold-Modell von Hämoglobin β (P68871)": ("P68871", "keins (Nummern der Struktur)", "E7V", "P68871"),
    "HIV-Protease: aktives Zentrum mit Hemmstoff (1HVR)": ("1HVR", "keins (Nummern der Struktur)", "D25N, I50V", ""),
}
struktur, protein, positionen, uniprot = beispiele.get(BEISPIEL, (STRUKTUR, PROTEIN, POSITIONEN, UNIPROT))
if protein in O.BEISPIEL_SEQUENZEN:
    protein = O.beispiel_fasta(protein)
elif protein.startswith("keins"):
    protein = None
print(f"Lade {struktur} … (große Strukturen brauchen ein paar Sekunden)")
display(HTML(O.struktur_html(struktur, protein, positionen, uniprot, float(KONTAKT_ABSTAND))))

In [ ]:
#@title 8 🩸 Mutations-Labor: Mutation einbauen und Folgen sehen
#@markdown **Vorlage** wählen, oder „eigene Mutation“ und die Felder darunter ausfüllen. `c.1` ist das A des Startcodons ATG.
VORLAGE = "HbS (Sichelzellanämie): c.20A>T" #@param ["eigene Mutation", "HbS (Sichelzellanämie): c.20A>T", "HbC: c.19G>A", "β-Thalassämie Codon 39: c.118C>T", "β-Thalassämie Codon 41/42: c.126_129del (4 bp)", "Startcodon zerstört: c.2T>C", "3 Basen löschen (In-frame): c.22_24del", "1 Base einfügen (Frameshift): nach c.30"]
ART = "Base austauschen" #@param ["Base austauschen", "Basen löschen", "Basen einfügen"]
POSITION = 20 #@param {type:"integer"}
NEUE_BASE = "T" #@param ["A", "C", "G", "T"]
ANZAHL_LOESCHEN = 1 #@param {type:"integer"}
EINFUEGEN = "A" #@param {type:"string"}
#@markdown Leer = Hämoglobin-Gen HBB des Menschen. Sonst eigene Gen-Sequenz einfügen.
EIGENES_GEN = "" #@param {type:"string"}

# --- Werkzeuge bereitstellen (falls Zelle 0 noch nicht lief) ---
import os, sys, subprocess
if not os.path.exists("Bioinfo/.git"):
    subprocess.run(["git", "clone", "-q", "https://github.com/galaxytosst/Bioinfo"], check=True)
if os.path.abspath("Bioinfo") not in sys.path:
    sys.path.insert(0, os.path.abspath("Bioinfo"))
import oberflaeche as O, analyse
from IPython.display import HTML, display

vorlagen = dict(O.BEKANNTE_MUTATIONEN)
if vorlagen.get(VORLAGE):
    art, pos, base, anzahl, ins = vorlagen[VORLAGE]
else:
    art = {"Base austauschen": "snv", "Basen löschen": "del", "Basen einfügen": "ins"}[ART]
    pos, base, anzahl, ins = POSITION, NEUE_BASE, ANZAHL_LOESCHEN, EINFUEGEN
display(HTML(O.mutation_html(art, pos, base or "T", anzahl, ins or "A",
                             EIGENES_GEN if EIGENES_GEN.strip() else None)))

In [ ]:
#@title 9 🌐 Sequenzen von NCBI herunterladen (werden ein neues Projekt)
#@markdown Accession-Nummern mit Komma, z. B. von ncbi.nlm.nih.gov/nuccore
ACCESSIONS = "NC_045512.2, NC_004718.3, NC_019843.3" #@param {type:"string"}
PROJEKTNAME = "coronaviren" #@param {type:"string"}
ZIEL = "Eigene Daten (Colab-Ordner daten)" #@param ["Eigene Daten (Colab-Ordner daten)", "Google Drive"]
DRIVE_ORDNER = "/content/drive/MyDrive/Bioinformatik/daten" #@param {type:"string"}

# --- Werkzeuge bereitstellen (falls Zelle 0 noch nicht lief) ---
import os, sys, subprocess
if not os.path.exists("Bioinfo/.git"):
    subprocess.run(["git", "clone", "-q", "https://github.com/galaxytosst/Bioinfo"], check=True)
if os.path.abspath("Bioinfo") not in sys.path:
    sys.path.insert(0, os.path.abspath("Bioinfo"))
import oberflaeche as O, analyse
from IPython.display import HTML, display

def _ziel_basis(ziel, drive_ordner):
    if ziel == "Google Drive":
        if not os.path.isdir("/content/drive/MyDrive"):
            from google.colab import drive
            drive.mount("/content/drive")
        return drive_ordner.rstrip("/")
    return os.path.abspath("daten")

import re
import werkzeuge
nummern = [n for n in re.split(r"[\s,;]+", ACCESSIONS) if n]
ziel = os.path.join(_ziel_basis(ZIEL, DRIVE_ORDNER), O._projektname(PROJEKTNAME))
for acc, fehler in werkzeuge.lade_von_ncbi(nummern, ziel):
    print(f"{acc}: {'✅ geladen' if fehler is None else '⚠️ ' + fehler}")
print(f"Gespeichert in {ziel}. Jetzt Box 1 mit QUELLE = {ZIEL} und PROJEKTE = {O._projektname(PROJEKTNAME)}")

---
**Noch mehr:** Im Notebook `Werkzeuge.ipynb` stehen dieselben Werkzeuge als ausführlicher Code mit Erklärungen, zum Nachvollziehen und Ändern.